In [2]:
!pip install streamlit ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.4/323.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.3 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.6
    Uninstalling protobuf-5.29.6:
      Successfully uninstalled protobuf-5.29.6
  Attempting uninstall: absl-py
    Found existing installation: absl-py 1.4.0
    Uninstalling absl-py-1.4.0:
      Successfully uninstalled absl-py-1.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.

In [1]:
%%writefile web_simple_vrp.py
import streamlit as st
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp

# Đồ án cuối kỳ - Môn Optimization
# Đề tài: Simple Vehicle Routing Problem (VRP)
# Nhóm thực hiện: 2 sinh viên

def main():
    st.set_page_config(page_title="Demo VRP", layout="centered")
    st.title("🚛 Demo VRP - Đồ án Optimization")
    st.caption("Ứng dụng tìm đường đi ngắn nhất cho đội xe (Simple VRP)")
    st.markdown("---")

    if st.button("Chạy thuật toán OR-Tools"):
        # Data fix cứng để demo báo cáo
        # Ma trận khoảng cách giữa kho (0) và 4 khách hàng
        matrix_khoang_cach = [
            [0, 548, 776, 696, 582],
            [548, 0, 684, 308, 194],
            [776, 684, 0, 992, 878],
            [696, 308, 992, 0, 114],
            [582, 194, 878, 114, 0],
        ]
        so_xe = 2
        kho_xuat_phat = 0

        # Khai báo model theo docs của google
        manager = pywrapcp.RoutingIndexManager(len(matrix_khoang_cach), so_xe, kho_xuat_phat)
        routing = pywrapcp.RoutingModel(manager)

        # Hàm này để model tra cứu khoảng cách
        def distance_callback(from_index, to_index):
            node_a = manager.IndexToNode(from_index)
            node_b = manager.IndexToNode(to_index)
            return matrix_khoang_cach[node_a][node_b]

        transit_callback_index = routing.RegisterTransitCallback(distance_callback)
        routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

        # Setup tham số, xài heuristic đường rẻ nhất
        search_params = pywrapcp.DefaultRoutingSearchParameters()
        search_params.first_solution_strategy = routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC

        # Giải bài toán
        solution = routing.SolveWithParameters(search_params)

        # print("Debug: solution =", solution) # Cố tình để lại comment debug

        if solution:
            st.success(f"Đã giải xong! Tổng quãng đường: {solution.ObjectiveValue()} m")

            # Quét từng xe để in lộ trình lên UI
            for xe in range(so_xe):
                index = routing.Start(xe)
                route_str = f"**Xe {xe}:** Kho 0 "
                route_dist = 0

                while not routing.IsEnd(index):
                    prev_index = index
                    index = solution.Value(routing.NextVar(index))
                    route_dist += routing.GetArcCostForVehicle(prev_index, index, xe)
                    route_str += f" ➡️ Node {manager.IndexToNode(index)}"

                st.info(route_str)
                st.write(f"*Quãng đường xe này đi:* {route_dist} m")
        else:
            st.error("Lỗi: Thuật toán không tìm ra kết quả, check lại ràng buộc!")

if __name__ == '__main__':
    main()

Writing web_simple_vrp.py


In [ ]:
!wget -q -O - ipv4.icanhazip.com
!streamlit run web_simple_vrp.py & npx --yes localtunnel --port 8501

35.245.228.181
⠙⠹

⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋2026-06-16 12:46:15.831 Uvicorn server started on 0.0.0.0:8501
your url is: https://lazy-sides-knock.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.245.228.181:8501

